# Block 13 — LAB: Ensembles — Bagging, Built by Hand
### Advanced Machine Learning — M&T Bank

Loads `bank_marketing_features.csv` (unchanged since Day 1, Block 3). Same target/split as Block 12. No new CSV
comes out of this lab.

**Part 1 — The problem:** how much do two independently-bootstrapped decision trees actually disagree on the same
clients?

**Part 2 — The fix:** build bagging from scratch — average many bootstrapped trees — and watch what happens to
accuracy as the ensemble grows.

**Part 3 — The sanity check:** does the from-scratch version match `BaggingClassifier`? Does
`RandomForestClassifier`'s extra feature-subsampling randomness add anything on top of plain bagging here?

Look for `# TODO` — that's where your code goes. Each task has a hint; ask if you get stuck.


## Setup

In [ ]:
import warnings
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)

NAVY = "#251E4E"
PINK = "#FF1675"
GRAY = "#6b7280"
ORANGE = "#FF7B01"

# Data source: https://raw.githubusercontent.com/mithun-rk/mt-advml-course/main/Data/bank_marketing_features.csv
df = pd.read_csv("https://raw.githubusercontent.com/mithun-rk/mt-advml-course/main/Data/bank_marketing_features.csv", sep=";")
feature_cols = [c for c in df.columns if c not in ("education", "y", "duration")]
bool_cols = [c for c in df[feature_cols].columns if df[c].dtype == bool]
X = df[feature_cols].copy()
for c in bool_cols:
    X[c] = X[c].astype(int)
y = df["y"].values

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
Xtr_arr = Xtr.values
n = len(Xtr_arr)
print(f"Train: {n} rows. Test: {len(Xte)} rows.")


## Part 1 — How Much Do Two Trees Actually Disagree?

### TODO 1.1 — Fit two trees on two different bootstrap samples

For each of two trees: draw a bootstrap sample (`np.random.RandomState(seed).choice(n, size=n, replace=True)`),
fit a `DecisionTreeClassifier(random_state=seed)` on `Xtr_arr[idx], ytr[idx]`. Use seed `0` for tree 1, seed `1`
for tree 2.


In [ ]:
# TODO: rng1 = np.random.RandomState(0); idx1 = rng1.choice(n, size=n, replace=True)
# TODO: dt1 = DecisionTreeClassifier(random_state=0).fit(Xtr_arr[idx1], ytr[idx1])
# TODO: same for dt2 with seed 1


### TODO 1.2 — Measure how much they disagree

Get `predict()` (0.5-threshold classification) and `predict_proba()[:, 1]` from both trees on `Xte`. Compute: the
fraction of test rows where the two trees' classifications agree, and the correlation between their predicted
probabilities (`np.corrcoef`).


In [ ]:
# TODO: pred1, pred2 = dt1.predict(Xte), dt2.predict(Xte)
# TODO: proba1, proba2 = dt1.predict_proba(Xte)[:, 1], dt2.predict_proba(Xte)[:, 1]
# TODO: agree_rate = ...
# TODO: corr = np.corrcoef(proba1, proba2)[0, 1]
# TODO: print both


### TODO 1.3 — Plot it

Scatter `proba1` (x-axis) against `proba2` (y-axis) for every test client, with a diagonal reference line at
`y=x`. If the two trees agreed perfectly, every point would sit on that line.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
# TODO: ax.scatter(proba1, proba2, s=8, alpha=0.3, color=NAVY)
ax.plot([0, 1], [0, 1], color=PINK, linestyle="--", linewidth=1.5)
ax.set_xlabel("Tree 1 predicted probability")
ax.set_ylabel("Tree 2 predicted probability")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()


**Question to answer:** how spread out are the points around the diagonal? Does that match what the agreement
rate and correlation numbers from 1.2 told you?


## Part 2 — Does Averaging Actually Fix It?

### TODO 2.1 — Write a manual bagging function

`manual_bagging_proba(B, seed0=0)`: for `b` in `range(B)`, draw a bootstrap sample with seed `seed0 + b`, fit a
`DecisionTreeClassifier(random_state=seed0 + b)` on it, get its `predict_proba(Xte)[:, 1]`, and accumulate. Return
the *average* predicted probability across all `B` trees.


In [ ]:
def manual_bagging_proba(B, seed0=0):
    probas = np.zeros(len(Xte))
    for b in range(B):
        # TODO: bootstrap sample with seed (seed0 + b)
        # TODO: fit a tree, accumulate its predict_proba(Xte)[:, 1] into `probas`
        pass
    return probas / B


### TODO 2.2 — Sweep B and measure test ROC-AUC

For `B` in `[1, 5, 25, 100]`: call `manual_bagging_proba(B)`, score it with `roc_auc_score(yte, proba)`, and time
how long each takes. Collect into a small table.


In [ ]:
bagging_rows = []
for B in [1, 5, 25, 100]:
    t0 = time.time()
    # TODO: proba = manual_bagging_proba(B)
    # TODO: auc = roc_auc_score(yte, proba)
    # TODO: append a dict with B, auc, and elapsed time to bagging_rows
    pass

pd.DataFrame(bagging_rows)


### TODO 2.3 — Plot AUC vs. B

Line plot, `B` on a log-scaled x-axis, test ROC-AUC on the y-axis.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
# TODO: Bs = [...]; aucs = [...]
# TODO: ax.plot(Bs, aucs, marker="o", color=PINK, linewidth=2.5)
ax.set_xscale("log")
ax.set_xlabel("Number of bootstrapped trees (B), log scale")
ax.set_ylabel("Test ROC-AUC")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()


**Question to answer:** where do the returns start to diminish? Compare `B=1`'s AUC to Block 6's logistic
benchmark (0.7958) and to `B=100`'s AUC — what does that say about how much a single overfit tree was actually
worth on its own?


## Part 3 — Sanity Check: Does This Match scikit-learn?

### TODO 3.1 — Fit `BaggingClassifier` and `RandomForestClassifier`

Both with `n_estimators=100, random_state=42, n_jobs=-1`. `BaggingClassifier` needs
`estimator=DecisionTreeClassifier(random_state=42)` passed explicitly. Score both on the test set.


In [ ]:
# TODO: bag = BaggingClassifier(estimator=DecisionTreeClassifier(random_state=42), n_estimators=100, random_state=42, n_jobs=-1)
# TODO: bag.fit(Xtr, ytr); bag_auc = roc_auc_score(yte, bag.predict_proba(Xte)[:, 1])

# TODO: rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
# TODO: rf.fit(Xtr, ytr); rf_auc = roc_auc_score(yte, rf.predict_proba(Xte)[:, 1])

# TODO: put your manual B=100 result (from Part 2) alongside bag_auc and rf_auc in one small table


**Questions to answer:**
1. Does your from-scratch B=100 result land close to `BaggingClassifier`'s? (It should — same idea, different
   implementation.)
2. Does `RandomForestClassifier`'s extra per-split feature subsampling meaningfully beat plain bagging on this
   dataset? If not, is that a bug, or a real finding — and what would make feature subsampling matter more?


## Wrap-Up

Write 2-3 sentences: how much did two independently-bootstrapped trees disagree, how much did averaging many of
them buy you, and how does this connect to Block 5's k-fold cross-validation (same "averaging reduces variance"
idea, applied to a different problem)?
